In [20]:
import pandas as pd

# 1. Define file paths
path_gecon = r"C:\Users\sanjay\OneDrive\Desktop\scratchpad_archive\002.kiva_loans_datasets\cleaned_dataframes\cleaned_gecon_v4.csv"
path_coords = r"C:\Users\sanjay\OneDrive\Desktop\scratchpad_archive\002.kiva_loans_datasets\cleaned_dataframes\cleaned_loan_coords.csv"
path_locations = r"C:\Users\sanjay\OneDrive\Desktop\scratchpad_archive\002.kiva_loans_datasets\cleaned_dataframes\cleaned_locations.csv"
path_bridge = r"C:\Users\sanjay\OneDrive\Desktop\scratchpad_archive\002.kiva_loans_datasets\cleaned_dataframes\cleaned_loans_lenders.csv"
path_lenders = r"C:\Users\sanjay\OneDrive\Desktop\scratchpad_archive\002.kiva_loans_datasets\cleaned_dataframes\cleaned_lenders.csv"

# 2. Load datasets
df_gecon = pd.read_csv(path_gecon)
df_coords = pd.read_csv(path_coords)
df_locations = pd.read_csv(path_locations)
df_bridge = pd.read_csv(path_bridge)
df_lenders = pd.read_csv(path_lenders)



In [21]:
# 3. Standardize column names
for df in [df_gecon, df_coords, df_locations, df_bridge, df_lenders]:
    df.columns = df.columns.str.strip().str.lower()

# Rename latitude so all datasets use the same key
df_coords.rename(columns={"latitude": "lat"}, inplace=True)

# 4. Clean merge keys
# Country
df_gecon["country"] = (
    df_gecon["country"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df_locations["country"] = (
    df_locations["country"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Loan IDs
df_coords["loan_id"] = (
    df_coords["loan_id"]
    .astype(str)
    .str.strip()
)

df_bridge["loan_id"] = (
    df_bridge["loan_id"]
    .astype(str)
    .str.strip()
)

# Lender IDs (bridge uses "lenders", lender table uses "permanent_name")
df_bridge["lenders"] = (
    df_bridge["lenders"]
    .astype(str)
    .str.strip()
)

df_lenders["permanent_name"] = (
    df_lenders["permanent_name"]
    .astype(str)
    .str.strip()
)

# Optional but recommended for coordinate joins
df_coords["lat"] = df_coords["lat"].round(6)
df_gecon["lat"] = df_gecon["lat"].round(6)

df_coords["longitude"] = df_coords["longitude"].round(6)
df_gecon["longitude"] = df_gecon["longitude"].round(6)



In [22]:
# Force correct naming first
df_coords = df_coords.rename(columns={'latitude': 'lat', 'longitude': 'long'})
df_gecon = df_gecon.rename(columns={'lat': 'lat', 'longitude': 'long'})

# Round the values to create the common grid key
df_coords['grid_lat'] = df_coords['lat'].round()
df_coords['grid_long'] = df_coords['long'].round()

df_gecon['grid_lat'] = df_gecon['lat'].round()
df_gecon['grid_long'] = df_gecon['long'].round()

# Merge on the newly created grid keys
df_master = df_coords.merge(
    df_gecon, 
    on=['grid_lat', 'grid_long'], 
    how='left'
)

print(f"Merge successful. New shape: {df_master.shape}")

Merge successful. New shape: (1216487, 27)


In [23]:
df_master

,loan_id,lat_x,long_x,grid_lat,grid_long,area,country,lat_y,long_y,long_name,...,popgpw_2000_40,popgpw_2005_40,ppp1990_40,ppp1995_40,ppp2000_40,ppp2005_40,quality,rig,quality_revision,date of last
0,657307,8.162411,123.774120,8.0,124.0,6767.0,philippines,8.0,124.0,Republic of the Philippines,...,1.897114e+06,1.970570e+06,3.090808,3.144826,3.771740,4.359498,1.0,0.552,1.0,2006-12-30
1,673156,8.162411,123.774120,8.0,124.0,6767.0,philippines,8.0,124.0,Republic of the Philippines,...,1.897114e+06,1.970570e+06,3.090808,3.144826,3.771740,4.359498,1.0,0.552,1.0,2006-12-30
2,888943,8.162411,123.774120,8.0,124.0,6767.0,philippines,8.0,124.0,Republic of the Philippines,...,1.897114e+06,1.970570e+06,3.090808,3.144826,3.771740,4.359498,1.0,0.552,1.0,2006-12-30
3,613323,8.162411,123.774120,8.0,124.0,6767.0,philippines,8.0,124.0,Republic of the Philippines,...,1.897114e+06,1.970570e+06,3.090808,3.144826,3.771740,4.359498,1.0,0.552,1.0,2006-12-30
4,777937,8.162411,123.774120,8.0,124.0,6767.0,philippines,8.0,124.0,Republic of the Philippines,...,1.897114e+06,1.970570e+06,3.090808,3.144826,3.771740,4.359498,1.0,0.552,1.0,2006-12-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1216482,155137,16.485432,121.151330,16.0,121.0,11543.0,philippines,16.0,121.0,Republic of the Philippines,...,1.342195e+06,1.479829e+06,1.666748,1.847828,2.182598,2.677725,1.0,0.970,1.0,2006-12-30
1216483,162583,10.037150,-84.158338,10.0,-84.0,5706.0,costa rica,10.0,-84.0,Republic of Costa Rica,...,3.392241e+05,4.054059e+05,1.400455,1.992925,2.753578,3.650401,1.0,0.468,0.0,2006-12-30
1216484,172304,8.828198,125.100705,9.0,125.0,4267.0,philippines,9.0,125.0,Republic of the Philippines,...,6.438108e+05,6.961419e+05,0.666238,0.724371,0.867168,1.043372,1.0,0.349,1.0,2006-12-30
1216485,852449,-14.240377,33.768533,-14.0,34.0,5334.0,malawi,-14.0,34.0,Republic of Malawi,...,5.806898e+05,6.768444e+05,0.276639,0.329197,0.409127,0.438434,1.0,0.444,1.0,2006-12-30


In [24]:
print(df_master['mer1990_40'].isna().sum())

3327


In [25]:
print(df_master.info())

<class 'pandas.DataFrame'>
RangeIndex: 1216487 entries, 0 to 1216486
Data columns (total 27 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   loan_id           1216487 non-null  str    
 1   lat_x             1216487 non-null  float64
 2   long_x            1216487 non-null  float64
 3   grid_lat          1216487 non-null  float64
 4   grid_long         1216487 non-null  float64
 5   area              1213160 non-null  float64
 6   country           1213160 non-null  str    
 7   lat_y             1213160 non-null  float64
 8   long_y            1213160 non-null  float64
 9   long_name         1213160 non-null  str    
 10  mer1990_40        1213160 non-null  float64
 11  mer1995_40        1213160 non-null  float64
 12  mer2000_40        1213160 non-null  float64
 13  mer2005_40        1213160 non-null  float64
 14  newcountryid      1213160 non-null  float64
 15  popgpw_1990_40    1213160 non-null  float64
 16  popgpw_1995

In [26]:
# Initialize a clean structural copy
df_master_clean = df_master.copy()

# 1. Identify rows that are exact duplicates across all columns
duplicate_count = df_master.duplicated().sum()
print(f"Total exact duplicate rows found: {duplicate_count}")

# 2. Check for potential 'semantic' duplicates based on unique identifiers
# If a loan_id appears more than once, something went wrong in the merge
loan_id_duplicates = df_master['loan_id'].duplicated().sum()
print(f"Duplicate loan_ids found: {loan_id_duplicates}")

# 3. Remove exact duplicates
df_master_clean = df_master_clean.drop_duplicates()

# 4. Final verification
print(f"Final shape after duplicate removal: {df_master_clean.shape}")

Total exact duplicate rows found: 0
Duplicate loan_ids found: 0
Final shape after duplicate removal: (1216487, 27)


In [ ]:
df_master_clean.to_csv(r"C:\Users\sanjay\OneDrive\Desktop\scratchpad_archive\002.kiva_loans_datasets\cleaned_dataframes\kiva_master_consolidated.csv", index=False)

print("Final cleaned consolidated kiva loan datasets is successfully exported!")

Final Cleaned Consolidated Master Dataset is Successfully Exported!
